# Secretory Lineage scRNA-seq Analysis

**Purpose**: Exploratory analysis of secretory epithelial cell lineage

**Input**: Secretory_Lineage_filtered.h5ad

**Analysis Steps**:
1. Data loading and structure inspection
2. QC metrics visualization
3. Dimensionality reduction and clustering visualization
4. Marker gene expression analysis
5. Cell composition analysis
6. Basic differential expression framework

In [ ]:
# ===== Configuration =====
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting parameters
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.set_figure_params(scanpy=True, dpi=100, dpi_save=300, 
                      vector_friendly=True, fontsize=12)

# File paths
INPUT_FILE = '/home/h2048/data/py/0114/cnmf_v1.3_fixed_optimized_k/Secretory_Lineage_filtered.h5ad'
OUTPUT_DIR = '/home/h2048/data/py/0114/cnmf_v1.3_fixed_optimized_k/analysis_output/'

# Create output directory if not exists
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Random seed for reproducibility
np.random.seed(42)

## 1. Data Loading and Basic Information

In [ ]:
# Load data
print("=" * 80)
print("LOADING DATA")
print("=" * 80)

adata = sc.read_h5ad(INPUT_FILE)

print(f"\n✓ Data loaded successfully")
print(f"  Shape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
if hasattr(adata.X, 'data'):
    print(f"  Memory usage: {adata.X.data.nbytes / 1e9:.2f} GB (sparse)")
else:
    print(f"  Memory usage: {adata.X.nbytes / 1e9:.2f} GB (dense)")

## 2. Data Structure Inspection

In [ ]:
# Check data structure
print("\n" + "=" * 80)
print("DATA STRUCTURE")
print("=" * 80)

print(f"\n.X (main matrix): {type(adata.X).__name__}")
print(f"  Data type: {adata.X.dtype}")

print(f"\n.layers available: {list(adata.layers.keys())}")
print(f".obsm keys (embeddings): {list(adata.obsm.keys())}")
print(f".obsp keys (graphs): {list(adata.obsp.keys())}")
print(f".uns keys (metadata): {list(adata.uns.keys())}")

if adata.raw is not None:
    print(f"\n.raw exists: {adata.raw.shape[0]:,} cells × {adata.raw.shape[1]:,} genes")
else:
    print("\n.raw: Not available")

In [ ]:
# Cell metadata (obs)
print("\n" + "=" * 80)
print("CELL METADATA (.obs)")
print("=" * 80)
print(f"\nColumns: {adata.obs.columns.tolist()}")
print(f"\nFirst 5 rows:")
adata.obs.head()

In [ ]:
# Gene metadata (var)
print("\n" + "=" * 80)
print("GENE METADATA (.var)")
print("=" * 80)
print(f"\nColumns: {adata.var.columns.tolist()}")
print(f"\nFirst 5 rows:")
adata.var.head()

## 3. Summary Statistics

In [ ]:
# Cell counts by key variables
print("\n" + "=" * 80)
print("CELL COUNTS SUMMARY")
print("=" * 80)

# Check for common grouping variables
group_vars = ['cell_type', 'celltype', 'cluster', 'leiden', 'louvain', 
              'disease_status', 'condition', 'sample', 'batch', 'Sample']

for var in group_vars:
    if var in adata.obs.columns:
        print(f"\n{var}:")
        print(adata.obs[var].value_counts().sort_index())
        print(f"  Total unique values: {adata.obs[var].nunique()}")

## 4. QC Metrics Visualization

In [ ]:
# Check for QC metrics
qc_metrics = ['n_genes', 'n_counts', 'percent_mito', 'pct_counts_mt', 
              'total_counts', 'n_genes_by_counts']

available_qc = [m for m in qc_metrics if m in adata.obs.columns]

if available_qc:
    print(f"Available QC metrics: {available_qc}")
    
    # Violin plots for QC metrics
    n_metrics = len(available_qc)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5*n_metrics, 4))
    if n_metrics == 1:
        axes = [axes]
    
    for i, metric in enumerate(available_qc):
        axes[i].violinplot([adata.obs[metric].values], positions=[0], 
                          showmeans=True, showmedians=True)
        axes[i].set_ylabel(metric)
        axes[i].set_title(f'{metric} Distribution')
        axes[i].set_xticks([])
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}qc_metrics_violin.pdf', bbox_inches='tight')
    plt.show()
else:
    print("⚠️ No standard QC metrics found in .obs")

## 5. Dimensionality Reduction Visualization

In [ ]:
# Check available embeddings
print("Available embeddings:", list(adata.obsm.keys()))

# UMAP visualization (if available)
if 'X_umap' in adata.obsm:
    # Find categorical variables for coloring
    cat_vars = adata.obs.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Prioritize common variables
    priority_vars = ['cell_type', 'celltype', 'cluster', 'leiden', 'louvain', 
                     'disease_status', 'condition', 'sample']
    
    color_by = []
    for var in priority_vars:
        if var in cat_vars:
            color_by.append(var)
    
    # Add remaining categorical variables (limit to first 6 total)
    for var in cat_vars:
        if var not in color_by and len(color_by) < 6:
            color_by.append(var)
    
    if color_by:
        sc.pl.umap(adata, color=color_by, ncols=3, frameon=False,
                  save='_categorical_overview.pdf')
    else:
        sc.pl.umap(adata, frameon=False, save='_basic.pdf')
else:
    print("⚠️ UMAP not found. Need to compute embeddings.")

## 6. Marker Gene Expression Analysis

In [ ]:
# Define secretory cell markers
secretory_markers = {
    'Secretory_General': ['SCGB1A1', 'SCGB3A1', 'MUC5B'],
    'Goblet': ['MUC5AC', 'TFF3', 'SPDEF'],
    'Serous': ['LTF', 'LYZ', 'DMBT1'],
    'Club': ['SCGB1A1', 'SCGB3A2', 'CYP2F1'],
}

# Flatten marker list and check availability
all_markers = []
for markers in secretory_markers.values():
    all_markers.extend(markers)
all_markers = list(set(all_markers))

# Check which markers are available
if adata.raw is not None:
    available_markers = [g for g in all_markers if g in adata.raw.var_names]
else:
    available_markers = [g for g in all_markers if g in adata.var_names]

print(f"\nRequested {len(all_markers)} markers, {len(available_markers)} available:")
print(available_markers)

missing_markers = set(all_markers) - set(available_markers)
if missing_markers:
    print(f"\n⚠️ Missing markers: {missing_markers}")

In [ ]:
# UMAP visualization of marker genes (if available)
if 'X_umap' in adata.obsm and available_markers:
    # Plot in batches of 6
    for i in range(0, len(available_markers), 6):
        batch_markers = available_markers[i:i+6]
        sc.pl.umap(adata, color=batch_markers, ncols=3, 
                  use_raw=True if adata.raw else False,
                  frameon=False, cmap='RdBu_r',
                  save=f'_markers_batch{i//6+1}.pdf')

In [ ]:
# Dotplot of marker genes (if clustering available)
cluster_var = None
for var in ['cell_type', 'celltype', 'cluster', 'leiden', 'louvain']:
    if var in adata.obs.columns:
        cluster_var = var
        break

if cluster_var and available_markers:
    sc.pl.dotplot(adata, available_markers, groupby=cluster_var,
                 use_raw=True if adata.raw else False,
                 dendrogram=True, save='_secretory_markers.pdf')

## 7. Cell Composition Analysis

In [ ]:
# Cell composition by sample and cell type
sample_var = None
for var in ['sample', 'Sample', 'batch', 'orig.ident']:
    if var in adata.obs.columns:
        sample_var = var
        break

if cluster_var and sample_var:
    # Create composition table
    composition = pd.crosstab(adata.obs[sample_var], adata.obs[cluster_var])
    
    # Proportion table
    composition_prop = composition.div(composition.sum(axis=1), axis=0)
    
    print("\nCell composition (absolute counts):")
    print(composition)
    
    print("\nCell composition (proportions):")
    print(composition_prop.round(3))
    
    # Heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(composition_prop, annot=True, fmt='.2f', cmap='YlOrRd',
               cbar_kws={'label': 'Proportion'})
    plt.title(f'Cell Type Composition by {sample_var}')
    plt.xlabel(cluster_var)
    plt.ylabel(sample_var)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}composition_heatmap.pdf', bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Missing sample or cluster information for composition analysis")

## 8. Differential Expression (Framework)

In [ ]:
# Check if DE results already exist
if 'rank_genes_groups' in adata.uns:
    print("✓ Differential expression results found in .uns['rank_genes_groups']")
    
    # Extract top DE genes
    de_results = sc.get.rank_genes_groups_df(adata, group=None)
    print(f"\nTop 10 DE genes per group:")
    print(de_results.head(10))
    
    # Heatmap of top DE genes
    if cluster_var:
        sc.pl.rank_genes_groups_heatmap(adata, n_genes=10, groupby=cluster_var,
                                       use_raw=True if adata.raw else False,
                                       show_gene_labels=True, dendrogram=True,
                                       save='_top_de_genes.pdf')
else:
    print("⚠️ No DE results found. Running basic DE analysis...")
    
    if cluster_var:
        # Run Wilcoxon test (fast for exploratory analysis)
        sc.tl.rank_genes_groups(adata, groupby=cluster_var, method='wilcoxon',
                               use_raw=True if adata.raw else False,
                               key_added='rank_genes_groups')
        
        print("\n✓ DE analysis completed")
        
        # Visualize
        sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False,
                               save='_volcano.pdf')
        
        sc.pl.rank_genes_groups_heatmap(adata, n_genes=10, groupby=cluster_var,
                                       use_raw=True if adata.raw else False,
                                       show_gene_labels=True, dendrogram=True,
                                       save='_top_de_genes.pdf')
    else:
        print("⚠️ No clustering variable found. Skipping DE analysis.")

## 9. Save Summary Report

In [ ]:
# Generate summary report
report = []
report.append("=" * 80)
report.append("SECRETORY LINEAGE ANALYSIS SUMMARY REPORT")
report.append("=" * 80)
report.append(f"\nInput file: {INPUT_FILE}")
report.append(f"Analysis date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report.append(f"\nDataset size: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

if cluster_var:
    report.append(f"\nCell type distribution (by {cluster_var}):")
    for ct, count in adata.obs[cluster_var].value_counts().items():
        pct = count / adata.shape[0] * 100
        report.append(f"  {ct}: {count:,} ({pct:.1f}%)")

if sample_var:
    report.append(f"\nSample distribution (by {sample_var}):")
    for s, count in adata.obs[sample_var].value_counts().items():
        report.append(f"  {s}: {count:,}")

report.append(f"\nAvailable embeddings: {list(adata.obsm.keys())}")
report.append(f"Available layers: {list(adata.layers.keys())}")

if available_markers:
    report.append(f"\nKey secretory markers detected ({len(available_markers)}/{len(all_markers)}):")
    report.append(f"  {', '.join(available_markers)}")

report_text = "\n".join(report)
print(report_text)

# Save report
with open(f'{OUTPUT_DIR}analysis_summary.txt', 'w') as f:
    f.write(report_text)

print(f"\n✓ Summary report saved to {OUTPUT_DIR}analysis_summary.txt")
print(f"✓ All figures saved to {OUTPUT_DIR}")

## 10. Optional: Save Processed Data

In [ ]:
# Optionally save the analyzed data with new results
# Uncomment if you want to save:
# output_file = OUTPUT_DIR + 'Secretory_Lineage_analyzed.h5ad'
# adata.write_h5ad(output_file, compression='gzip', compression_opts=9)
# print(f"✓ Analyzed data saved to {output_file}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)